# Router-Strict LoRA v0.2 Colab Notebook

第一輪只訓練工具路由與安全拒答，目標是穩定輸出 JSON tool call。不要把 `explainer_sft.jsonl` 或 `legacy_cleaned.jsonl` 混入這輪。

## Cell 1. 掛載 Google Drive 與設定專案路徑

Drive 內資料夾建議放在 `MyDrive/energy_lora_router_v02/`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/energy_lora_router_v02')
SCRIPT_PATH = PROJECT_DIR / 'colab_train_router_strict_lora.py'
DATA_DIR = PROJECT_DIR / 'data'

assert SCRIPT_PATH.exists(), f'Missing script: {SCRIPT_PATH}'
assert DATA_DIR.exists(), f'Missing data dir: {DATA_DIR}'

print('PROJECT_DIR =', PROJECT_DIR)
print('SCRIPT_PATH =', SCRIPT_PATH)

## Cell 2. 設定環境變數

如果 Hugging Face 模型需要授權，把 `HF_TOKEN` 填上。T4 請把 `LOAD_IN_4BIT` 設成 `true`。

In [ ]:
import os

os.environ['DRIVE_PROJECT_DIR'] = str(PROJECT_DIR)
os.environ.setdefault('MODEL_ID', 'google/gemma-4-e2b-it')
os.environ.setdefault('EXPERIMENT_NAME', 'gemma-e2b-energy-router-strict-v02')
os.environ.setdefault('GGUF_BASENAME', 'gemma-4-e2b-it-energy-router-v02-Q4_K_M.gguf')

# A100 建議值
os.environ.setdefault('LOAD_IN_4BIT', 'false')
os.environ.setdefault('TRAIN_BATCH_SIZE', '8')
os.environ.setdefault('GRAD_ACCUM_STEPS', '2')
os.environ.setdefault('LORA_R', '32')
os.environ.setdefault('NUM_TRAIN_EPOCHS', '3')
os.environ.setdefault('USE_WANDB', 'true')
os.environ.setdefault('EXPORT_GGUF', 'true')

# T4 改用這組：
# os.environ['LOAD_IN_4BIT'] = 'true'
# os.environ['TRAIN_BATCH_SIZE'] = '2'
# os.environ['GRAD_ACCUM_STEPS'] = '8'
# os.environ['LORA_R'] = '16'

# 若需要，取消註解並填入：
# os.environ['HF_TOKEN'] = 'hf_xxx'
# os.environ['WANDB_API_KEY'] = 'xxx'

print('MODEL_ID =', os.environ['MODEL_ID'])
print('LOAD_IN_4BIT =', os.environ['LOAD_IN_4BIT'])
print('TRAIN_BATCH_SIZE =', os.environ['TRAIN_BATCH_SIZE'])

## Cell 3. 安裝依賴

第一次跑需要安裝。若 Colab 安裝後要求重啟 runtime，重啟後從 Cell 1 開始跑，但這格可以略過。

In [ ]:
%pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q -U --no-deps trl peft accelerate bitsandbytes
%pip install -q -U datasets huggingface_hub sentencepiece protobuf wandb

os.environ['INSTALL_DEPS'] = 'false'

## Cell 4. 載入訓練工具模組

這格會從 Drive 讀 `colab_train_router_strict_lora.py`，但不會直接開始訓練。

In [ ]:
import importlib.util
import sys

spec = importlib.util.spec_from_file_location('router_lora_trainer', SCRIPT_PATH)
trainer = importlib.util.module_from_spec(spec)
sys.modules['router_lora_trainer'] = trainer
spec.loader.exec_module(trainer)

print('Loaded trainer from:', SCRIPT_PATH)
print('OUTPUT_DIR =', trainer.OUTPUT_DIR)

## Cell 5. 登入服務並檢查資料

這格會確認 `manifest.profile == router_strict`，並檢查 assistant target 都是 JSON。

In [ ]:
trainer.login_services()
trainer.validate_drive_files()

## Cell 6. 載入模型與 tokenizer

如果這格因 Hugging Face 授權失敗，先確認 `HF_TOKEN` 或 Gemma 授權。

In [ ]:
model, tokenizer = trainer.load_model_and_tokenizer()

## Cell 7. Render dataset

把 `messages` 轉成 SFTTrainer 吃的 `text` 欄位。

In [ ]:
rendered_dataset = trainer.build_datasets(tokenizer)

## Cell 8. 掛 LoRA adapter

這格會檢查沒有 vision/mmproj 參數被放進 LoRA targets。

In [ ]:
model = trainer.attach_lora(model)

## Cell 9. 開始訓練

A100 通常幾分鐘內完成；T4 會比較久。

In [ ]:
model = trainer.train(model, tokenizer, rendered_dataset)

## Cell 10. Smoke 評測

先跑 4 題 smoke，確認模型沒有整個歪掉。

In [ ]:
smoke_report = trainer.evaluate_split(model, tokenizer, trainer.SMOKE_FILE, 'smoke_after_train')
smoke_report

## Cell 11. 完整 validation 評測

第一輪目標：tool accuracy >= 80%，malformed JSON < 5%。

In [ ]:
val_report = trainer.evaluate_split(model, tokenizer, trainer.VAL_FILE, 'val_after_train')
val_report

## Cell 12. 匯出 merged model 與 GGUF

輸出會放在 Drive 的 `outputs/gemma_router_strict_v02/`。

In [ ]:
trainer.export_outputs(model, tokenizer)
print('Done. OUTPUT_DIR =', trainer.OUTPUT_DIR)
print('Final GGUF should be under:', trainer.OUTPUT_DIR / 'final_gguf')